In [1]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 14:29:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 14:29:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/08 14:29:19 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df = df.repartition(4)
df.write.parquet('tripdata_partitioned.parquet')

In [3]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [7]:
df.createOrReplaceTempView("trips")

df_15th = spark.sql("""
SELECT COUNT(*) AS trip_count
FROM trips
WHERE tpep_pickup_datetime >= '2025-11-15 00:00:00'
  AND tpep_pickup_datetime <  '2025-11-16 00:00:00'
""")

df_15th.show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



In [10]:
from pyspark.sql import functions as F

df_longest = df.select(
    (
        (F.unix_timestamp("tpep_dropoff_datetime") - 
         F.unix_timestamp("tpep_pickup_datetime")) / 3600
    ).alias("trip_hours")
)
df_longest.agg(F.max("trip_hours")).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [12]:
zone_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

zone_df.createOrReplaceTempView("zones")
df.createOrReplaceTempView("trips")

spark.sql("""
SELECT 
    z.Zone,
    COUNT(*) AS pickups
FROM trips t
JOIN zones z
ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY pickups ASC
""").show()

+--------------------+-------+
|                Zone|pickups|
+--------------------+-------+
|Governor's Island...|      1|
|Eltingville/Annad...|      1|
|       Arden Heights|      1|
|       Port Richmond|      3|
|       Rikers Island|      4|
|   Rossville/Woodrow|      4|
|         Great Kills|      4|
| Green-Wood Cemetery|      4|
|         Jamaica Bay|      5|
|         Westerleigh|     12|
|New Dorp/Midland ...|     14|
|       West Brighton|     14|
|             Oakwood|     14|
|        Crotona Park|     14|
|       Willets Point|     15|
|Breezy Point/Fort...|     16|
|Saint George/New ...|     17|
|       Broad Channel|     18|
|     Mariners Harbor|     21|
|Heartland Village...|     22|
+--------------------+-------+
only showing top 20 rows
